# RK1828 SenseVoice-Small 多语言语音识别 Demo


阿里 [SenseVoice-Small](https://github.com/FunAudioLLM/SenseVoice) 多语言 ASR，
输出里除了文字还**内联语种、情感和声音事件标签**（如 `<|zh|><|HAPPY|><|Speech|>`），
实测单条约 292ms。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型和示例音频（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 选择音频 |
| 4 | 识别（看富标签输出） |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("sensevoice-small-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/sensevoice/demo_x86_64.tar.gz"),
    ("sensevoice-small-rknn3-rk1828", "model/embedding_matrix.txt", "demos/sensevoice/model/embedding_matrix.txt"),
    ("sensevoice-small-rknn3-rk1828", "model/sensevoice_fix_10s.rknn", "demos/sensevoice/model/sensevoice_fix_10s.rknn"),
    ("sensevoice-small-rknn3-rk1828", "model/sensevoice_fix_10s.weight", "demos/sensevoice/model/sensevoice_fix_10s.weight"),
    ("sensevoice-small-rknn3-rk1828", "model/tokens.txt", "demos/sensevoice/model/tokens.txt"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "demos/sensevoice/model/zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "datasets/asr/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "datasets/asr/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "datasets/asr/long_en.wav"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "datasets/asr/sensevoice_zh.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "datasets/asr/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "datasets/asr/asr_example_hotword.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "datasets/asr/aligner_demo.wav"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

> **为什么这个 demo 用 C++ 而不是纯 Python？**
> 01-05（分类/检测/FastSAM/pose/分割）是单个模型一次前向，后处理简单，
> 所以用 Python（`rknn3.py`）逐步演示。而本 demo 的流水线复杂得多——
> 多模型串联、tokenizer、KV cache、滑窗/流式状态等，前后处理和编排逻辑
> 有几百上千行，重写 Python 版成本高且容易和官方实现不一致。
> 因此直接运行官方 `rknn3-model-zoo` 编译的 C++ demo，
> 源码见 [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> 的 `examples/` 目录。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/sensevoice/demo_x86_64.tar.gz", "demos/sensevoice")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 选择音频

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_wav = widgets.Dropdown(options=sorted(glob.glob("datasets/asr/*.wav")),
                         description="音频:", layout=widgets.Layout(width="60%"))
display(w_wav)
if any(o.endswith('/sensevoice_zh.wav') for o in w_wav.options):
    w_wav.value = [o for o in w_wav.options if o.endswith('/sensevoice_zh.wav')][0]
print("选好后继续往下运行 ▼")

先听一下这段音频：

In [ ]:
from nb_demo import show_audio

show_audio(w_wav.value)

## 4. 识别

In [ ]:
import os

DEMO = "demos/sensevoice"
wav = os.path.abspath(w_wav.value)
run_demo(["./rknn_sensevoice_demo",
          "model/sensevoice_fix_10s.rknn", "model/sensevoice_fix_10s.weight",
          wav, "0x1"], DEMO)

## 小结

- SenseVoice 把"语种/情感/事件"当成特殊 token 直接生成，一次推理得到转写 + 副语言信息。
- 模型输入固定 10s，短音频会自动补齐。